# Prerequisites

In [1]:
# get data for labs
!wget -nc -O around_the_world_in_80_days.txt https://www.gutenberg.org/ebooks/103.txt.utf-8

File ‘around_the_world_in_80_days.txt’ already there; not retrieving.


# 1. Word Count

Instructions:  
For each cell marked "double-click and add explanation here" please answer the question in your own words.  
In the section where you complete the code to perform basic nlp text cleaning and exploration tasks, the goal is to chain all of the transformations together in a single function. For learning and exploration purposes, it is acceptable to have each step seperate, but the last cell in this section should be one function with all transformations chained together.  
For steps c and f, it is acceptable to use your favorite chatbot to generate a list of common stop words (c) and punctuation (e) for use in the code. As these are common steps in nlp/text processing tasks, there are pleanty of libraries to help with this such as nltk, but there is no need to import extra dependencies for this lab unless you are already familiar with working with them.

In [2]:
# start a spark session and create spark context for making rdd
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("word_count") \
    .getOrCreate()

sc = spark.sparkContext

In [3]:
# Defind the rdd
rdd = sc.textFile('around_the_world_in_80_days.txt')

In [4]:
# view the first x lines of the rdd
rdd.take(20)

['The Project Gutenberg eBook of Around the World in Eighty Days',
 '    ',
 'This eBook is for the use of anyone anywhere in the United States and',
 'most other parts of the world at no cost and with almost no restrictions',
 'whatsoever. You may copy it, give it away or re-use it under the terms',
 'of the Project Gutenberg License included with this eBook or online',
 'at www.gutenberg.org. If you are not located in the United States,',
 'you will have to check the laws of the country where you are located',
 'before using this eBook.',
 '',
 'Title: Around the World in Eighty Days',
 '',
 'Author: Jules Verne',
 '',
 'Translator: George M. Towle',
 '',
 '',
 '        ',
 'Release date: January 1, 1994 [eBook #103]',
 '                Most recently updated: October 29, 2024']

In [5]:
# example lambda function
words = rdd.flatMap(lambda lines: lines.split(' '))

In [6]:
# Note and explain the output of the below command
words

PythonRDD[3] at RDD at PythonRDD.scala:59

It doesn't show the words, just something like `PythonRDD[3] at RDD at ...`.
`words` is only a reference to the RDD, not the data itself. !!!!!

This is because `flatMap` is a transformation and Spark is lazy: it only
remembers what it has to do and doesn't compute anything until we call an
action. !!!!!

In [7]:
# Note and explain the output of the following command, focusing on the difference with the
# above command
words.collect()[:50]  # we decided to display the first 50 words

['The',
 'Project',
 'Gutenberg',
 'eBook',
 'of',
 'Around',
 'the',
 'World',
 'in',
 'Eighty',
 'Days',
 '',
 '',
 '',
 '',
 '',
 'This',
 'eBook',
 'is',
 'for',
 'the',
 'use',
 'of',
 'anyone',
 'anywhere',
 'in',
 'the',
 'United',
 'States',
 'and',
 'most',
 'other',
 'parts',
 'of',
 'the',
 'world',
 'at',
 'no',
 'cost',
 'and',
 'with',
 'almost',
 'no',
 'restrictions',
 'whatsoever.',
 'You',
 'may',
 'copy',
 'it,',
 'give']

`collect()` is an action, so this time Spark actually runs the job and sends
back all the words to the driver as a Python list. We can see there are
empty strings, capital letters and punctuation stuck to some words. !!!!!

In [8]:
# nicer print
for w in words.collect()[:50]:  # again only the first 50 words
    print(w)

The
Project
Gutenberg
eBook
of
Around
the
World
in
Eighty
Days





This
eBook
is
for
the
use
of
anyone
anywhere
in
the
United
States
and
most
other
parts
of
the
world
at
no
cost
and
with
almost
no
restrictions
whatsoever.
You
may
copy
it,
give


In [9]:
# Print first x words
words.take(20)

['The',
 'Project',
 'Gutenberg',
 'eBook',
 'of',
 'Around',
 'the',
 'World',
 'in',
 'Eighty',
 'Days',
 '',
 '',
 '',
 '',
 '',
 'This',
 'eBook',
 'is',
 'for']

In [10]:
# Use cell magic command to help understand what the rdd.flatMap function is doing in the next cell.
# Insert a text/markdown cell and explain in your own words.
?rdd.flatMap # a checker !!!!!!!

Signature:
rdd.flatMap(
    f: Callable[[~T], Iterable[~U]],
    preservesPartitioning: bool = False,
) -> 'RDD[U]'
Docstring:
Return a new RDD by first applying a function to all elements of this
RDD, and then flattening the results.

.. versionadded:: 0.7.0

Parameters
----------
f : function
    a function to turn a T into a sequence of U
preservesPartitioning : bool, optional, default False
    indicates whether the input function preserves the partitioner,
    which should be False unless this is a pair RDD and the input
    function doesn't modify the keys

Returns
-------
:class:`RDD`
    a new :class:`RDD` by applying a function to all elements

See Also
--------
:meth:`RDD.map`
:meth:`RDD.mapPartitions`
:meth:`RDD.mapPartitionsWithIndex`
:meth:`RDD.mapPartitionsWithSplit`

Examples
--------
>>> rdd = sc.parallelize([2, 3, 4])
>>> sorted(rdd.flatMap(lambda x: range(1, x)).collect())
[1, 1, 1, 2, 2, 3]
>>> sorted(rdd.flatMap(lambda x: [(x, x), (x, x)]).collect())
[(2, 2), (2, 2)

`flatMap` applies the function to each line (here split, which gives a list
of words) and then flattens everything, so we get one element per word
instead of one list per line like with `map`. Then `map` turns each word
into `(word, 1)` so we can count them.

In [11]:
# Initialize a word counter by creating a tuple with word and cound of 1
words = rdd.flatMap(lambda lines: lines.split(' ')) \
                    .map(lambda word: (word, 1))

for w in words.collect()[:50]:  # first 50 tuples
    print(w)

('The', 1)
('Project', 1)
('Gutenberg', 1)
('eBook', 1)
('of', 1)
('Around', 1)
('the', 1)
('World', 1)
('in', 1)
('Eighty', 1)
('Days', 1)
('', 1)
('', 1)
('', 1)
('', 1)
('', 1)
('This', 1)
('eBook', 1)
('is', 1)
('for', 1)
('the', 1)
('use', 1)
('of', 1)
('anyone', 1)
('anywhere', 1)
('in', 1)
('the', 1)
('United', 1)
('States', 1)
('and', 1)
('most', 1)
('other', 1)
('parts', 1)
('of', 1)
('the', 1)
('world', 1)
('at', 1)
('no', 1)
('cost', 1)
('and', 1)
('with', 1)
('almost', 1)
('no', 1)
('restrictions', 1)
('whatsoever.', 1)
('You', 1)
('may', 1)
('copy', 1)
('it,', 1)
('give', 1)


In [12]:
# a. count the occurence of each word
word_counts = (rdd
               .flatMap(lambda line: line.split(' '))
               .map(lambda word: (word, 1))
               .reduceByKey(lambda a, b: a + b))
word_counts.take(20)

[('Gutenberg', 60),
 ('eBook', 6),
 ('of', 1875),
 ('Around', 4),
 ('', 2193),
 ('for', 407),
 ('use', 16),
 ('anyone', 6),
 ('United', 23),
 ('States', 10),
 ('and', 1793),
 ('most', 43),
 ('other', 59),
 ('world', 30),
 ('at', 576),
 ('no', 124),
 ('cost', 9),
 ('with', 550),
 ('almost', 19),
 ('restrictions', 2)]

In [13]:
# b. a common first step in text analysis, change all capital letters to lower case
lower_counts = (rdd
                .flatMap(lambda line: line.lower().split(' '))
                .map(lambda word: (word, 1))
                .reduceByKey(lambda a, b: a + b))
lower_counts.take(20)

[('of', 1926),
 ('around', 32),
 ('world', 35),
 ('eighty', 27),
 ('days', 46),
 ('', 2193),
 ('this', 341),
 ('for', 414),
 ('use', 19),
 ('anyone', 6),
 ('united', 27),
 ('states', 14),
 ('and', 1835),
 ('most', 45),
 ('other', 63),
 ('at', 645),
 ('no', 137),
 ('cost', 12),
 ('with', 562),
 ('almost', 19)]

In [14]:
# c. eliminate the stop words.
STOP_WORDS = {
    'a', 'about', 'above', 'after', 'again', 'against', 'all', 'am', 'an',
    'and', 'any', 'are', 'as', 'at', 'be', 'because', 'been', 'before',
    'being', 'below', 'between', 'both', 'but', 'by', 'can', 'could', 'did',
    'do', 'does', 'doing', 'down', 'during', 'each', 'few', 'for', 'from',
    'further', 'had', 'has', 'have', 'having', 'he', 'her', 'here', 'hers',
    'herself', 'him', 'himself', 'his', 'how', 'i', 'if', 'in', 'into', 'is',
    'it', 'its', 'itself', 'just', 'me', 'might', 'more', 'most', 'must',
    'my', 'myself', 'no', 'nor', 'not', 'now', 'of', 'off', 'on', 'once',
    'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own',
    'same', 'shall', 'she', 'should', 'so', 'some', 'such', 'than', 'that',
    'the', 'their', 'theirs', 'them', 'themselves', 'then', 'there',
    'these', 'they', 'this', 'those', 'through', 'to', 'too', 'under',
    'until', 'up', 'upon', 'very', 'was', 'we', 'were', 'what', 'when',
    'where', 'which', 'while', 'who', 'whom', 'why', 'will', 'with',
    'would', 'you', 'your', 'yours', 'yourself', 'yourselves',
    's', 't', 'd', 'll', 'm', 'o', 're', 've', 'y',
}

no_stop_counts = lower_counts.filter(lambda wc: wc[0] not in STOP_WORDS)
no_stop_counts.take(20)

[('around', 32),
 ('world', 35),
 ('eighty', 27),
 ('days', 46),
 ('', 2193),
 ('use', 19),
 ('anyone', 6),
 ('united', 27),
 ('states', 14),
 ('cost', 12),
 ('almost', 19),
 ('restrictions', 2),
 ('give', 18),
 ('re-use', 2),
 ('license', 12),
 ('online', 4),
 ('www.gutenberg.org.', 4),
 ('states,', 7),
 ('country', 18),
 ('using', 6)]

In [15]:
# d. sort in alphabetical order
alpha_sorted = no_stop_counts.sortByKey()
alpha_sorted.take(20)

[('', 2193),
 ('#103]', 1),
 ('#516,', 1),
 ('$5,000)', 1),
 ('&c.,', 1),
 ('($1', 1),
 ('(862)', 1),
 ('(a)', 1),
 ('(and', 1),
 ('(any', 1),
 ('(b)', 1),
 ('(c)', 1),
 ('(does', 1),
 ('(if', 1),
 ('(japan),', 1),
 ('(or', 3),
 ('(saturday,', 1),
 ('(sort', 1),
 ('(sunday)', 1),
 ('(trademark/copyright)', 1)]

In [16]:
# e. sort descending by word frequency
freq_sorted = no_stop_counts.sortBy(lambda wc: wc[1], ascending=False)
freq_sorted.take(20)

[('', 2193),
 ('mr.', 373),
 ('fogg', 365),
 ('phileas', 250),
 ('passepartout', 239),
 ('said', 157),
 ('one', 133),
 ('fogg,', 132),
 ('fix', 129),
 ('passepartout,', 121),
 ('“i', 115),
 ('two', 98),
 ('hundred', 92),
 ('replied', 89),
 ('project', 87),
 ('without', 86),
 ('thousand', 83),
 ('and,', 83),
 ('time', 82),
 ('made', 81)]

In [17]:
# f. remve punctuations and blank spaces
import string

PUNCTUATION = string.punctuation + '“”‘’«»—–…\ufeff'
PUNCT_TABLE = str.maketrans({char: ' ' for char in PUNCTUATION})

clean_counts = (freq_sorted
                .flatMap(lambda wc: [(w, wc[1]) for w
                                     in wc[0].translate(PUNCT_TABLE).split()])
                .filter(lambda wc: wc[0] not in STOP_WORDS)
                .reduceByKey(lambda a, b: a + b)
                .sortBy(lambda wc: wc[1], ascending=False))
clean_counts.take(20)

[('fogg', 646),
 ('passepartout', 424),
 ('mr', 391),
 ('phileas', 256),
 ('fix', 256),
 ('said', 194),
 ('one', 174),
 ('aouda', 136),
 ('master', 129),
 ('time', 126),
 ('train', 119),
 ('two', 108),
 ('sir', 103),
 ('project', 99),
 ('hundred', 98),
 ('twenty', 97),
 ('well', 96),
 ('replied', 93),
 ('steamer', 91),
 ('hours', 90)]

In [18]:
# all the transformations chained together in a single function
def word_count(lines, stop_words=STOP_WORDS):
    return (lines
            .map(lambda line: line.lower())
            .map(lambda line: line.translate(PUNCT_TABLE))
            .flatMap(lambda line: line.split())
            .filter(lambda word: word not in stop_words)
            .map(lambda word: (word, 1))
            .reduceByKey(lambda a, b: a + b)
            .sortBy(lambda wc: wc[1], ascending=False))


word_count(rdd).take(20)

[('fogg', 646),
 ('passepartout', 424),
 ('mr', 391),
 ('phileas', 256),
 ('fix', 256),
 ('said', 194),
 ('one', 174),
 ('aouda', 136),
 ('master', 129),
 ('time', 126),
 ('train', 119),
 ('two', 108),
 ('sir', 103),
 ('project', 99),
 ('hundred', 98),
 ('twenty', 97),
 ('well', 96),
 ('replied', 93),
 ('steamer', 91),
 ('hours', 90)]

# 2. What does the following cell block do?
Comment the code below line by line after the provided hash-tag. You should be able to explain each line while respecting the pep8 style guide of 79 characters or less per line!

In [19]:
# we Create an RDD of tuples (name,age
dataRDD = sc.parallelize([("Brooke", 20), ("Denny", 31), ("Jules", 30),
("TD", 35), ("Brooke", 25)])

# Try to undestand what this code does (line by line)
agesRDD = (dataRDD
  # (name, age) -> (name, (age, 1)): the 1 counts the people with this name
  .map(lambda x: (x[0], (x[1], 1)))
  # per name,add the ages together and the counters together
    
  # -> (name, (sum_of_ages, nb_of_people))
  .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
  # (name, (sum, count)) -> (name, sum / count) = average age per name
  .map(lambda x: (x[0], x[1][0]/x[1][1])))

This code computes the average age for each name. Each tuple becomes
`(name, (age, 1))`, then `reduceByKey` adds the ages and the 1s for the same
name, and the last `map` divides the total age by the number of people.
For example Brooke: (20 + 25) / 2 = 22.5.

## 3. Function timing.

- write a simple python timer function for seeing how quickly your rdd runs as written. change the order of the steps in order to make the rdd run as optimally as possible


In [25]:
# Code here
import time


def timer(func, *args):
    """Return the result of func(*args) and its duration in seconds."""
    start = time.perf_counter()
    result = func(*args)
    return result, time.perf_counter() - start


def exercise_order(lines):
    # we did the order of part 1
    return (lines
            .flatMap(lambda line: line.split(' '))
            .map(lambda word: (word, 1))
            .reduceByKey(lambda a, b: a + b)
            .map(lambda wc: (wc[0].lower(), wc[1]))
            .reduceByKey(lambda a, b: a + b)
            .filter(lambda wc: wc[0] not in STOP_WORDS)
            .sortByKey()
            .sortBy(lambda wc: wc[1], ascending=False)
            .flatMap(lambda wc: [(w, wc[1]) for w
                                 in wc[0].translate(PUNCT_TABLE).split()])
            .filter(lambda wc: wc[0] not in STOP_WORDS)
            .reduceByKey(lambda a, b: a + b)
            .sortBy(lambda wc: wc[1], ascending=False)
            .collect())


def sort_before_count(lines):
    # we clean first but sort th words before counting them
    return (lines
            .map(lambda line: line.lower().translate(PUNCT_TABLE))
            .flatMap(lambda line: line.split())
            .filter(lambda word: word not in STOP_WORDS)
            .sortBy(lambda word: word)
            .map(lambda word: (word, 1))
            .reduceByKey(lambda a, b: a + b)
            .sortBy(lambda wc: wc[1], ascending=False)
            .collect())


def filter_after_count(lines):
    #we clean first, remove the stop words after the count
    return (lines
            .map(lambda line: line.lower().translate(PUNCT_TABLE))
            .flatMap(lambda line: line.split())
            .map(lambda word: (word, 1))
            .reduceByKey(lambda a, b: a + b)
            .filter(lambda wc: wc[0] not in STOP_WORDS)
            .sortBy(lambda wc: wc[1], ascending=False)
            .collect())


def clean_filter_count_sort(lines):
    # function of part 1, clean, filter, count once, sort once 
    return word_count(lines).collect()


for pipeline in [exercise_order, sort_before_count, filter_after_count,
                 clean_filter_count_sort]:
    _, duration = timer(pipeline, rdd)
    print(f'{pipeline.__name__:25} {duration:.3f} s')

exercise_order            2.083 s
sort_before_count         0.784 s
filter_after_count        0.414 s
clean_filter_count_sort   0.456 s


The order of part 1 (a to f) is the slowest because it does `reduceByKey`
and sorting several times, and each one is a shuffle.

## 4. Text Comparison

- perform eda on the original french version of the [book](https://www.gutenberg.org/ebooks/46541.txt.utf-8) and compare the two

In [21]:
# Code here
!wget -nc -O le_tour_du_monde_en_80_jours.txt https://www.gutenberg.org/ebooks/46541.txt.utf-8

File ‘le_tour_du_monde_en_80_jours.txt’ already there; not retrieving.


In [22]:
fr_rdd = sc.textFile('le_tour_du_monde_en_80_jours.txt')

STOP_WORDS_FR = {
    'a', 'à', 'ai', 'aient', 'ait', 'alors', 'au', 'aucun', 'aussi',
    'autre', 'aux', 'avait', 'avaient', 'avec', 'avoir', 'c', 'ce', 'cela',
    'celle', 'celui', 'ces', 'cet', 'cette', 'ceux', 'chez', 'comme', 'd',
    'dans', 'de', 'des', 'donc', 'dont', 'du', 'elle', 'elles', 'en',
    'encore', 'est', 'et', 'étaient', 'était', 'été', 'être', 'eu', 'eût',
    'fait', 'fut', 'fût', 'il', 'ils', 'j', 'je', 'l', 'la', 'là', 'le',
    'les', 'leur', 'leurs', 'lui', 'm', 'ma', 'mais', 'me', 'même', 'mes',
    'moi', 'mon', 'n', 'ne', 'ni', 'nos', 'notre', 'nous', 'on', 'ont',
    'ou', 'où', 'par', 'pas', 'peu', 'plus', 'pour', 'qu', 'quand', 'que',
    'quel', 'quelle', 'qui', 's', 'sa', 'sans', 'se', 'ses', 'si', 'son',
    'sont', 'sous', 'sur', 't', 'ta', 'te', 'tes', 'toi', 'ton', 'tous',
    'tout', 'toute', 'toutes', 'très', 'tu', 'un', 'une', 'vos', 'votre',
    'vous', 'y',
}


def all_words(lines):
    return lines.flatMap(lambda line: line.lower()
                         .translate(PUNCT_TABLE).split())


# EDA
for name, lines in [('english', rdd), ('french', fr_rdd)]:
    words = all_words(lines)
    print(f'{name:8} lines: {lines.count():6}  words: {words.count():6}  '
          f'distinct words: {words.distinct().count():5}')

en_counts = word_count(rdd)
fr_counts = word_count(fr_rdd, STOP_WORDS_FR)
print('\nenglish top 15:', en_counts.take(15))
print('\nfrench top 15 :', fr_counts.take(15))

english  lines:   8312  words:  67509  distinct words:  7129
french   lines:   9969  words:  76738  distinct words:  9599

english top 15: [('fogg', 646), ('passepartout', 424), ('mr', 391), ('phileas', 256), ('fix', 256), ('said', 194), ('one', 174), ('aouda', 136), ('master', 129), ('time', 126), ('train', 119), ('two', 108), ('sir', 103), ('project', 99), ('hundred', 98)]

french top 15 : [('fogg', 689), ('passepartout', 460), ('phileas', 332), ('mr', 287), ('fix', 287), ('heures', 243), ('répondit', 215), ('bien', 196), ('the', 190), ('dit', 183), ('deux', 153), ('monsieur', 145), ('aouda', 136), ('quelques', 136), ('après', 133)]


In [24]:
# query: compare the counts of the words present in both versions
comparison = (en_counts
              .join(fr_counts)       # (word, (english_count, french_count))
              .sortBy(lambda x: x[1][0] + x[1][1], ascending=False))

comparison.take(20)

[('fogg', (646, 689)),
 ('passepartout', (424, 460)),
 ('mr', (391, 287)),
 ('phileas', (256, 332)),
 ('fix', (256, 287)),
 ('aouda', (136, 136)),
 ('train', (119, 115)),
 ('project', (99, 88)),
 ('one', (174, 2)),
 ('monsieur', (29, 145)),
 ('gutenberg', (81, 81)),
 ('sir', (103, 55)),
 ('hong', (63, 67)),
 ('kong', (63, 67)),
 ('gentleman', (42, 87)),
 ('bombay', (61, 61)),
 ('moment', (50, 70)),
 ('minutes', (64, 51)),
 ('steamer', (91, 19)),
 ('francis', (54, 53))]

The French version is longer 76,738 words vs 67,509 and has more
different words 9,599 vs 7,129, probably because French has more forms for
the same word. The main characters appear about
the same number of times